# Sourcing AI · Google Colab 启动笔记本

按顺序执行下面 4 个代码单元即可。运行结束后会在最后一行打印公网地址，
用浏览器打开它，输入账号密码登录。

> 免费版 Colab 长时间无操作会被回收实例，公网链接随之失效，重新执行最后一格会生成新链接。

## 1. 拉取代码

把 `REPO_URL` 换成你自己的仓库地址。如果是私有仓库，用
`https://<用户名>:<PersonalAccessToken>@github.com/<用户名>/<仓库>.git` 的形式。

In [ ]:
REPO_URL = "https://github.com/devinlin2022/sourcing-ai.git"
PROJECT = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

%cd /content
!rm -rf {PROJECT}
!git clone {REPO_URL}
%cd {PROJECT}
!ls

## 2. 安装依赖

Colab 已自带大部分包，这一步主要是补 DuckDB / Gradio / lxml 等，约 1-2 分钟。

In [ ]:
!pip install -q -r requirements.txt
!python -c "import duckdb, gradio, fastapi; print('deps ok')"

## 3. 配置密码与 ngrok token

* 密码：登录用，自己设一个。
* ngrok token：到 <https://dashboard.ngrok.com/get-started/your-authtoken> 免费注册后复制，
  没有它就只能在笔记本内部访问 `127.0.0.1`，浏览器打不开。

> 想持久化数据（Colab 重启不丢），把下面 `USE_DRIVE` 改成 `True`。

In [ ]:
import os
from getpass import getpass

USE_DRIVE = False

os.environ["ADMIN_USER"] = "admin"
os.environ["ADMIN_PASSWORD"] = getpass("设置登录密码: ") or "admin123"
os.environ["NGROK_AUTHTOKEN"] = getpass("粘贴 ngrok authtoken: ")

# 可选：硅基流动 DeepSeek，用于「AI 助手」页签
# os.environ["SILICONFLOW_API_KEY"] = getpass("SiliconFlow API Key（可回车跳过）: ")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/sourcing_ai", exist_ok=True)
    os.environ["DB_PATH"] = "/content/drive/MyDrive/sourcing_ai/sourcing.duckdb"

print("配置完成，密码已设置，token 长度:", len(os.environ["NGROK_AUTHTOKEN"]))

## 4. 启动服务

等待打印 `Sourcing AI 已启动: https://...`，点击该链接即可。
首次进入先在「数据管理」页签上传 CSV / RapidRatings xlsx。

In [ ]:
from tools.colab import boot

server = boot(port=7860)

## 常见问题

**Q: 提示 ngrok 启动失败？**  
A: token 没填或已失效。重新执行第 3 格设置 token，再执行第 4 格。免费账号同一时间只允许 1 条隧道。

**Q: 上传 100MB+ 的 CSV 很慢？**  
A: 浏览器上传 Colab 不稳定，建议先传到 Google Drive，然后在笔记本里复制：
```python
!cp "/content/drive/MyDrive/数据/Cost Exhibit.csv" /content/data/
```
再在界面上传 `/content/data/` 下的文件。

**Q: 数据丢了？**  
A: Colab 磁盘是临时的。把 `USE_DRIVE` 设为 `True` 再启动，数据库会写到 Drive。

**Q: 行情看板刷不出来？**  
A: TradingEconomics 被限流时会自动切世界银行月度兜底，界面上会显示当前来源与数据日期；过几分钟再点一次刷新。